Segmenting Consumers of Bath Soap - Module 5 - Samuel Scott

## Part 1: Business Situation
### The Agency
CRISA is an Asian market research agency that tracks consumer purchase behavior across consumer
goods, both durable and nondurable. In one major project it follows numerous product categories, such
as detergents, and dozens of brands within each. To track behavior, CRISA built household panels in
more than 100 cities and towns across India, covering most of the urban market. The households were
drawn by stratified sampling on socioeconomic status and market to keep the sample representative, and
a subset of 600 households is analyzed here.
CRISA keeps both transaction data (one row per transaction) and household data (one row per
household). For each household it maintains demographics (updated annually), possession of durable
goods (updated annually, from which an affluence index is computed), and purchase data by product
category and brand (updated monthly). Its clients fall into two groups: advertising agencies that subscribe
to the database and advise their own clients on advertising and promotion, and consumer goods
manufacturers that monitor their market share.

### The Segmentation Problem
CRISA has traditionally segmented markets by purchaser demographics. It now wants to segment on two
sets of variables tied more directly to the purchase process and to brand loyalty: purchase behavior
(volume, frequency, susceptibility to discounts, and brand loyalty) and basis of purchase (price and selling
proposition).
Better segmentation along these lines would reveal which demographic attributes go with which purchase
behaviors and degrees of loyalty, so promotion budgets could be deployed more effectively. It would let
CRISA's client, a firm called IMRB, design more cost-effective promotions aimed at the right segments,
run multiple promotions targeted at different segments at different times of year, and build more effective
customer reward systems to strengthen brand loyalty.

### Measuring Brand Loyalty
Several variables capture brand loyalty, and they capture different things. The number of distinct brands a
household buys is one measure. But a household that buys one or two brands briefly and then settles on
a third for a long stretch is not the same as one that constantly switches among three brands, so how
often a household switches (its brand runs) is a second measure. A third is concentration of spend: a
household that puts 90 percent of its money on a single brand is more loyal than one that spreads its
spending evenly across brands. All three can be derived from the purchase-summary and brand-share
variables, and how you combine them is one of the key modeling decisions in this case.

In [ ]:
import heapq
import random 
from collections import defaultdict
import matplotlib.pyplot as plt
import mlba
import pandas as pd
import surprise
from mlxtend.frequent_patterns import apriori, association_rules
from surprise import Dataset, Reader, KNNBasic
from surprise.model_selection import train_test_split
from sklearn.model_selection import train_test_split
import numpy as np
import seaborn as sns
from adjustText import adjust_text
from pandas.plotting import parallel_coordinates
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from sklearn import preprocessing
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import OneHotEncoder
import warnings
warnings.filterwarnings('ignore')

In [ ]:
bathsoap = pd.read_csv('C:/Users/samsc/Desktop/ADS-505/BathSoapHousehold.csv')
bathsoap.head()

,Member id,SEC,FEH,MT,SEX,AGE,EDU,HS,CHILD,CS,...,PropCat 6,PropCat 7,PropCat 8,PropCat 9,PropCat 10,PropCat 11,PropCat 12,PropCat 13,PropCat 14,PropCat 15
0,1010010,4,3,10,1,4,4,2,4,1,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.028037,0.0,0.130841,0.339564
1,1010020,3,2,10,2,2,4,4,2,1,...,0.347048,0.026834,0.016100,0.014311,0.0,0.059034,0.000000,0.0,0.080501,0.000000
2,1014020,2,3,10,2,4,5,6,4,1,...,0.121212,0.033550,0.010823,0.008658,0.0,0.000000,0.016234,0.0,0.561688,0.003247
3,1014030,4,0,0,0,4,0,0,5,0,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.600000,0.000000
4,1014190,4,1,10,2,3,4,4,3,1,...,0.000000,0.000000,0.048193,0.000000,0.0,0.000000,0.000000,0.0,0.144578,0.000000


In [ ]:
#bathsoap.info()

### Question 1 - Segment the Households with k-Means
Use k-means clustering to identify clusters of households based on each of the following variable sets.

#### 1.1 Cluster on Purchase Behavior
Cluster households using the variables that describe purchase behavior, including brand loyalty.

In [ ]:
loyalty = bathsoap[['Br. Cd. 57, 144', 'Br. Cd. 55', 'Br. Cd. 272','Br. Cd. 286', 'Br. Cd. 24', 'Br. Cd. 481', 'Br. Cd. 352', 'Br. Cd. 5']]
bathsoap['Loyalty'] = loyalty.max(axis=1) #This column is used to determine loyalty as a understandable business value

purchase_behavior = bathsoap[['No. of Brands','Brand Runs', 'Total Volume', 'No. of  Trans', 'Value', 'Trans / Brand Runs', 'Vol/Tran', 'Avg. Price ',
                             'Pur Vol No Promo - %', 'Pur Vol Promo 6 %', 'Pur Vol Other Promo %', 'Loyalty']]
purchase_behavior_norm = purchase_behavior.apply(preprocessing.scale, axis=0)
kmeans_purch_behav = KMeans(n_clusters=3,random_state=1).fit(purchase_behavior_norm)

#### 1.2 Cluster on Basis of Purchase
Cluster households using the variables that describe the basis for purchase (price and selling
proposition).

In [ ]:
basis_purchase = bathsoap[['Others 999','Pr Cat 1','Pr Cat 2','Pr Cat 3','Pr Cat 4','PropCat 5','PropCat 6', 'PropCat 7', 'PropCat 8', 
                          'PropCat 9', 'PropCat 10', 'PropCat 11', 'PropCat 12', 'PropCat 13', 'PropCat 14', 'PropCat 15', 'Loyalty']]
basis_purchase_norm = basis_purchase.apply(preprocessing.scale, axis=0)
kmeans_basis_purch = KMeans(n_clusters=3,random_state=1).fit(basis_purchase_norm)

#### 1.3 Cluster on Both
Cluster households using both the purchase-behavior and basis-of-purchase variables together.

In [ ]:
basis_purchase = basis_purchase.drop(columns=['Loyalty']) #drop one of the two loyalty columns
comb = pd.concat([purchase_behavior, basis_purchase], axis=1)
kmeans_comb = KMeans(n_clusters=3,random_state=1).fit(comb)
basis_purchase['Loyalty'] = loyalty.max(axis=1) #Add it back for analysis

### Question 2 - Choose and Profile the Best Segmentation
From the three clustering solutions in Question 1, select the segmentation you think is most useful, and
comment on the characteristics of its clusters across three lenses: demographics, brand loyalty, and basis
of purchase. This profile is what guides the development of advertising and promotional campaigns, so
describe each segment in terms a campaign planner could act on.

In [ ]:
#Question 2 was initially confusing for me and I used AI to help me understand what segmentation meant and what most useful meant.
#The answer I got was something along the lines of which cluster showed the most separation amongst values in the features. 
#1.Loyalty Scores:

#Purchase Behavior Cluster
purchase_behavior['Purchase_Cluster_Labels'] = kmeans_purch_behav.labels_
purchase_behav_loyalty = purchase_behavior.groupby('Purchase_Cluster_Labels')['Loyalty'].mean().round(2)
#print(purchase_behav_loyalty)

#Basis Purchase Cluster
basis_purchase['Basis_Cluster_Labels'] = kmeans_basis_purch.labels_
purchase_basis_loyalty = basis_purchase.groupby('Basis_Cluster_Labels')['Loyalty'].mean().round(2)
print(purchase_basis_loyalty)

#Comb Purchase Cluster
comb['Comb_Cluster_Labels'] = kmeans_comb.labels_
comb_loyalty = comb.groupby('Comb_Cluster_Labels')['Loyalty'].mean().round(2)
#print(comb_loyalty)

Basis_Cluster_Labels
0    0.37
1    0.20
2    0.75
Name: Loyalty, dtype: float64


In [ ]:
#2.Basis Purchase Scores:
basis_basis_score = basis_purchase.groupby('Basis_Cluster_Labels').mean().round(2)
print(basis_basis_score)

                      Others 999  Pr Cat 1  Pr Cat 2  Pr Cat 3  Pr Cat 4  \
Basis_Cluster_Labels                                                       
0                           0.51      0.16      0.65      0.06      0.13   
1                           0.71      0.64      0.32      0.02      0.02   
2                           0.16      0.06      0.13      0.78      0.02   

                      PropCat 5  PropCat 6  PropCat 7  PropCat 8  PropCat 9  \
Basis_Cluster_Labels                                                          
0                          0.60       0.08       0.09       0.07       0.03   
1                          0.30       0.14       0.16       0.13       0.04   
2                          0.11       0.06       0.01       0.01       0.02   

                      PropCat 10  PropCat 11  PropCat 12  PropCat 13  \
Basis_Cluster_Labels                                                   
0                           0.01        0.04        0.00        0.00   
1      

In [ ]:
#3.Demographics: 
#Used value counts here because the features here are discrete categorical variables
demographics = bathsoap[['SEC', 'FEH', 'MT', 'SEX', 'EDU', 'HS', 'CS', 'Affluence Index']]
demographics['Basis_Cluster_Labels'] = kmeans_basis_purch.labels_
demographics_sec = demographics.groupby('Basis_Cluster_Labels')['SEC'].value_counts(normalize=True).round(2)
print(demographics_sec)
demographics_feh = demographics.groupby('Basis_Cluster_Labels')['FEH'].value_counts(normalize=True).round(2)
print(demographics_feh)
demographics_mt = demographics.groupby('Basis_Cluster_Labels')['MT'].value_counts(normalize=True).round(2)
print(demographics_mt)
demographics_sex = demographics.groupby('Basis_Cluster_Labels')['SEX'].value_counts(normalize=True).round(2)
print(demographics_sex)
demographics_edu = demographics.groupby('Basis_Cluster_Labels')['EDU'].value_counts(normalize=True).round(2)
print(demographics_edu)
demographics_hs = demographics.groupby('Basis_Cluster_Labels')['HS'].value_counts(normalize=True).round(2)
print(demographics_hs)
demographics_cs = demographics.groupby('Basis_Cluster_Labels')['CS'].value_counts(normalize=True).round(2)
print(demographics_cs)
demographics_ai = demographics.groupby('Basis_Cluster_Labels')['Affluence Index'].mean().round(2)
print(demographics_ai)

Basis_Cluster_Labels  SEC
0                     3      0.28
                      4      0.27
                      1      0.23
                      2      0.22
1                     1      0.41
                      2      0.34
                      3      0.21
                      4      0.04
2                     4      0.61
                      2      0.18
                      3      0.18
                      1      0.03
Name: proportion, dtype: float64
Basis_Cluster_Labels  FEH
0                     3      0.57
                      1      0.28
                      0      0.08
                      2      0.06
1                     3      0.48
                      1      0.33
                      0      0.15
                      2      0.04
2                     3      0.62
                      0      0.20
                      1      0.13
                      2      0.05
Name: proportion, dtype: float64
Basis_Cluster_Labels  MT
0                     10    0.57
        

The features that were selected to be used on the cluster was the basis of purchase features because it these are the features that are most reflective on business revenue since they are features that explore percent of volume purchases. The cluster or segmentation that represented the data best through demographics, loyalty, and basis of purchase is cluster 1 as it had the affluent index of these customers showing 20.41 which was the highest of the 3 clusters. Cluster 1 had the lowest loyalty value score at 0.20 out of the three clusters so the campaign would probably be interested in convincing this population of people to subscribe or show more loyalty to their campaign in order to gain more revenue. Exploring the basis of purchase features the highest scores for Cluster 1 are Others 999 and Price Cat 1, which shows which Category the advertising team should focus their promotional campaigns on. When exploring some demographics that stood out the most were SEX which had an 0.81 value for female and CS or television availability with a score of 0.76.

### Question 3 - Build a Classifier for the Segments
Develop a model that classifies households into the segments you selected in Question 2. Because this
would most likely be used to target direct-mail promotions, define one market segment as the success
(positive) class in your classification model, and evaluate the model with that targeting use in mind.

In [ ]:
bathsoap['Basis_Cluster_Labels'] = kmeans_basis_purch.labels_

#identify demographic variables to one hot encode vs ordinal encoder
nominal_cols = ['FEH', 'MT', 'SEX', 'CHILD', 'CS']

nom_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
nom_data = bathsoap[nominal_cols]
nom_feat = pd.DataFrame(nom_encoder.fit_transform(nom_data),
                        columns=nom_encoder.get_feature_names_out(nominal_cols),
                        index=nom_data.index)
#nom_feat
new_bathsoap = pd.concat([bathsoap, nom_feat], axis=1)
new_bathsoap = new_bathsoap.drop(columns=['FEH', 'MT', 'SEX', 'CHILD', 'CS', 'Member id'])
new_bathsoap.head()

,SEC,AGE,EDU,HS,Affluence Index,No. of Brands,Brand Runs,Total Volume,No. of Trans,Value,...,SEX_1,SEX_2,CHILD_1,CHILD_2,CHILD_3,CHILD_4,CHILD_5,CS_0,CS_1,CS_2
0,4,4,4,2,2,3,17,8025,24,818.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1,3,2,4,4,19,5,25,13975,40,1681.5,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,2,4,5,6,23,5,37,23100,63,1950.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,4,4,0,0,0,2,4,1500,4,114.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
4,4,3,4,4,10,3,6,8300,13,591.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


In [ ]:
#LDA model

#bathsoap['Basis_Cluster_Labels'].value_counts()

mask2=new_bathsoap[new_bathsoap['Basis_Cluster_Labels'] == 2] #all clusters that are labeled 2
mask0=new_bathsoap[new_bathsoap['Basis_Cluster_Labels'] == 0] #all clusters that are labeled 0
#print(type(mask0)) #dataframe has index feature we can extract values from mask through it

new_bathsoap['Classification'] = 1 #positive class

new_bathsoap.loc[mask0.index, 'Classification'] = 0 #negative class
new_bathsoap.loc[mask2.index, 'Classification'] = 0 #negative class

#bathsoap['Classification'].value_counts()

outcome = 'Classification'
predictors = new_bathsoap.drop(columns=['Basis_Cluster_Labels', 'Classification'])
predictors = predictors.columns

In [ ]:
X = new_bathsoap[predictors]
y = new_bathsoap[outcome]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=1)

lda = LinearDiscriminantAnalysis(priors=[0.50,0.50]) #using priors to avoid class imbalance
lda.fit(X_train, y_train)

y_pred = lda.predict(X_test)
(y_pred == y_test).mean()

np.float64(0.9625)

In [ ]:
mlba.classificationSummary(y_true=y_test, y_pred=y_pred)

Confusion Matrix (Accuracy 0.9625)

       Prediction
Actual   0   1
     0 171   7
     1   2  60


The LDA model is able to accurately predict the customers the business wants to target base on the clusters. The business would want to use this model because it will be able to target customers who are more affluent andare going to pay more. Because the loyalty is low for customers in cluster one the business would spend money to advertise and campaign to these individuals so they are able to get a larger revenue. 